### **CC57 - MACHINE LEARNING - APRENDIZAJE POR REFUERZO: Metodo Q-Learning**

Vamos a implementar un ejemplo básico de la tecnica **Q-Learning** utilizando el entorno **"FrozenLake-v1"** de **OpenAI** Gym.

El ejemplo de **Frozen Lake** es un problema clásico de aprendizaje por refuerzo, donde el objetivo es enseñar a un agente a navegar por un lago congelado y alcanzar la meta sin caer en el hielo. El agente está en un lago congelado y necesita recuperar un elemento y donde algunas partes del lago están congeladas y otras son agujeros (si entra en ellos, muere).

**Acciones:** IZQUIERDA: 0 ABAJO = 1 DERECHA = 2 ARRIBA = 3

En este ejemplo, utilizamos el algoritmo **Q-learning** para entrenar a un agente para que navegue por el entorno de FrozenLake.  El agente aprende gradualmente qué acciones son mejores en cada lugar y, con el tiempo, puede llegar a la meta de forma eficaz.

**Descripción del problema:** Un agente debe moverse en un lago helado desde el inicio hasta el objetivo sin caer en los agujeros.

**Objetivo:** Demuestra cómo un agente puede aprender a navegar en un entorno simple utilizando **Q-Learning**.

**En el paso 1 y 2 se realiza la Inicialización:**
   - Importa las librerías necesarias (gym, numpy, matplotlib)
   - Crea el entorno Frozen Lake, donde el agente debe llegar a la meta sin caer en los agujeros.
   - Crea una tabla Q, que es como una memoria donde el agente almacena la "calidad" de realizar ciertas acciones en ciertos lugares. Al principio, la tabla está llena de ceros porque el agente no sabe nada.


####**Paso #1:  Importar librerías y crear entorno**

Utilizamos gym para crear el entorno "FrozenLake-v1".

In [19]:
import gym
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


En el contexto del código que proporcionaste, `new_step_api=False` se refiere a una **opción para configurar la interfaz de interacción con el entorno de aprendizaje por refuerzo (RL) de OpenAI Gym.**

**En términos más sencillos:**

* **OpenAI Gym** es una biblioteca que proporciona entornos de RL predefinidos para experimentar con algoritmos de aprendizaje por refuerzo.
* **La API de pasos (step API)** es la forma en que un agente interactúa con el entorno. El agente realiza una acción, el entorno responde con una nueva observación, una recompensa y un indicador de si el episodio ha terminado.
* **`new_step_api`** es una bandera que controla qué versión de la API de pasos se utiliza. `new_step_api=False` indica que se debe utilizar la versión anterior de la API.


**¿Por qué usar `new_step_api=False` en este caso?**

El código que proporcionaste es un ejemplo de Q-learning, un algoritmo de aprendizaje por refuerzo clásico. En algunos casos, los algoritmos de RL antiguos pueden no ser compatibles con las nuevas API. Al establecer `new_step_api=False`, el código se asegura de utilizar la versión de la API que es compatible con ese algoritmo específico.

**En resumen:**

`new_step_api=False` es una opción de configuración que indica que se debe utilizar la versión anterior de la API de pasos en OpenAI Gym. Se usa para garantizar la compatibilidad con los algoritmos de RL más antiguos, como el Q-learning en el ejemplo proporcionado.

In [20]:
# Crear el entorno
env = gym.make("FrozenLake-v1", new_step_api=False)

####**Paso #2: Inicializar tabla Q**
Se inicializa una matriz de estados por acciones llena de ceros.

In [21]:
# Inicializar la tabla Q con ceros
Q = np.zeros((env.observation_space.n, env.action_space.n))


La inicialización de la tabla Q con ceros se realiza porque representa la estimación inicial de la función de valor de acción (Q-value). En otras palabras, al principio no se sabe nada sobre la calidad de tomar una acción en un estado específico.

**Aquí la de explicacion por qué se inicia con ceros:**

1. **Representación inicial:** La tabla Q se usa para almacenar la función Q, que estima la recompensa total que el agente puede esperar si realiza una acción específica en un estado particular. Inicializar con ceros indica que al inicio no hay ninguna preferencia por una acción sobre otra en ningún estado.

2. **Exploración:**  Al comenzar con valores de Q nulos (ceros), el agente está más propenso a explorar diferentes acciones y estados, ya que todas parecen igualmente prometedoras (o no prometedoras). Esto es importante porque, si se empieza con una tabla Q con valores fijos, es posible que el agente se quede atascado en un camino subóptimo, lo que impedirá que explore otras opciones con un mejor rendimiento a largo plazo.

3. **Aprendizaje gradual:** A medida que el agente interactúa con el entorno y se realizan nuevas observaciones, la tabla Q se actualizará gradualmente con mejores estimaciones. Cada vez que se realiza una acción y se recibe una recompensa, los valores de Q se modifican para representar mejor la calidad de las acciones y estados.


En resumen, la inicialización con ceros es una forma de asegurar que el agente comienza su aprendizaje con una mente "limpia" y sin preferencia previa, lo que le permite explorar el entorno para adquirir conocimiento gradualmente y mejorar la calidad de sus decisiones a lo largo del tiempo.


####**Paso #3: Configuracion de Parámetros de aprendizaje**

A continuacion, el detalle de los parametros utilizados para el entrenamiento del agente.

In [22]:
# Parámetros de aprendizaje
"""
- Define parámetros importantes para el aprendizaje:
- `alpha`:  Tasa de aprendizaje (qué tan rápido aprende el agente).
- `gamma`:  Factor de descuento (cuánto valoramos las recompensas futuras).
- `epsilon`: Tasa de exploración (cuánto exploramos acciones aleatorias).
- `episodes`: Número de veces que el agente juega el juego para aprender.
- `max_steps_per_episode`: Número máximo de movimientos que puede hacer el agente en cada partida.
"""

alpha = 0.1  # Tasa de aprendizaje
gamma = 0.99  # Factor de descuento
epsilon = 0.1  # Tasa de exploración inicial
episodes = 2000  # Número de episodios
max_steps_per_episode = 100  # Número máximo de pasos por episodio

**1. `alpha` (Tasa de aprendizaje):**

* **Qué es:** `alpha` representa la tasa de aprendizaje, que es un valor entre 0 y 1.
* **Función:** Determina cuánto se actualizan los valores de Q en cada paso de aprendizaje. Un `alpha` alto significa que el agente aprenderá rápidamente de nuevas experiencias, pero también podría volverse inestable, mientras que un `alpha` bajo significa un aprendizaje más gradual y estable.
* **En el código:** `alpha = 0.1`, lo que indica un aprendizaje moderado.

**2. `gamma` (Factor de descuento):**

* **Qué es:** `gamma` es un valor entre 0 y 1 que representa el factor de descuento.
* **Función:** Determina la importancia de las recompensas futuras en comparación con las recompensas inmediatas. Un `gamma` alto le da más importancia a las recompensas futuras, mientras que un `gamma` bajo enfatiza principalmente las recompensas inmediatas.
* **En el código:** `gamma = 0.99`, significa que se da una alta importancia a las recompensas futuras. Esto es coherente con el objetivo del agente de encontrar el camino más largo a la meta.

**3. `epsilon` (Tasa de exploración):**

* **Qué es:** `epsilon` representa la tasa de exploración.
* **Función:** Define la probabilidad de que el agente tome una acción aleatoria en lugar de una acción basada en la tabla Q. La exploración es importante para descubrir nuevas acciones y estados que podrían conducir a mejores recompensas a largo plazo.
* **En el código:** `epsilon = 0.1`, indica que el 10% de las veces, el agente explorará tomando una acción aleatoria. Esto permite que el agente encuentre nuevas opciones que podrían llevarlo a un desempeño mejor.

**4. `episodes` (Número de episodios):**

* **Qué es:** `episodes` indica el número total de episodios que el agente realizará durante el entrenamiento.
* **Función:** Un episodio es una secuencia de pasos que termina cuando el agente alcanza el objetivo o entra en un agujero en el juego. El agente se entrena a través de una serie de episodios.
* **En el código:** `episodes = 2000`, el agente realizará 2000 episodios para aprender.

**5. `max_steps_per_episode` (Número máximo de pasos por episodio):**

* **Qué es:** `max_steps_per_episode` es un límite del número de pasos que el agente puede realizar en cada episodio.
* **Función:** Se utiliza para evitar que el aprendizaje se prolongue de forma indefinida. Si el agente no llega al objetivo o cae en un agujero dentro del número máximo de pasos, el episodio se detiene.
* **En el código:** `max_steps_per_episode = 100`, se limita el episodio a 100 pasos.

**Resumen:**

Estos parámetros son vitales para el proceso de entrenamiento Q-learning.

* **`alpha` y `gamma`** determinan cómo el agente aprende de las recompensas y actualiza sus valores de Q.
* **`epsilon`** controla el equilibrio entre la exploración y la explotación.
* **`episodes` y `max_steps_per_episode`** dictan la duración y la frecuencia del entrenamiento.

En general, la configuración de estos parámetros afecta significativamente la velocidad de aprendizaje, la estabilidad y la capacidad del agente para encontrar la mejor estrategia en el entorno.

In [23]:
# Mapeo de acciones a direcciones
# Esta lista facilita la verificación y comprensión de qué acción se está tomando.
actions = ["IZQUIERDA", "ABAJO", "DERECHA", "ARRIBA"]

#### **Paso #4:  Función epsilon-greedy**

Selecciona acciones basadas en una política epsilon-greedy.

**Función epsilon-greedy:**
   - Esta función determina qué acción toma el agente.
   - A veces, elige una acción al azar (exploración) para descubrir nuevas opciones.
   - Otras veces, elige la acción que según la tabla Q parece mejor en ese momento (explotación).

In [24]:
# Función epsilon-greedy para seleccionar acciones
def epsilon_greedy(state, epsilon):
    if np.random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    else:
        return np.argmax(Q[state, :])

#### **Paso #5: Entrenamiento del agente**
Iteramos sobre episodios y actualizamos la tabla Q con la fórmula de **Q-Learning**.

In [25]:
# Entrenamiento del agente
rewards = []

for episode in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0
    steps = 0

    while not done and steps < max_steps_per_episode:
        action = epsilon_greedy(state, epsilon)
        next_state, reward, done, _ = env.step(action)

        # Debug: Print state, action, reward, next_state
        # se imprimen el episodio, el paso, el estado actual, la acción tomada (usando el nombre descriptivo),
        # la recompensa recibida y el siguiente estado.
        # Esto proporciona una visión clara de cómo el agente interactúa con el entorno y cómo se actualiza la tabla Q.
        print(f"Episode: {episode}, Step: {steps}, State: {state}, Action: {actions[action]}, Reward: {reward}, Next State: {next_state}")

        # Recompensa adicional por llegar al estado final
        if done and reward == 0:
            reward = -1  # Penalizar si cae en un agujero
        elif done and reward == 1:
            reward = 1  # Recompensa si llega al objetivo

        # Actualización de la tabla Q
        Q[state, action] = Q[state, action] + alpha * (reward + gamma * np.max(Q[next_state, :]) - Q[state, action])

        # Debug: Print Q-table
        print(f"Q[{state}, {action}]: {Q[state, action]}")

        state = next_state
        total_reward += reward
        steps += 1

    rewards.append(total_reward)
    epsilon = max(epsilon * 0.01, 0.01)  # Reducir epsilon gradualmente
    # Al reducir epsilon gradualmente se asegura que el agente explora suficiente al principio y explota más hacia el final del entrenamiento.

Se han truncado las últimas 5000 líneas del flujo de salida.
Episode: 1940, Step: 41, State: 0, Action: IZQUIERDA, Reward: 0.0, Next State: 0
Q[0, 0]: 0.30456359611386347
Episode: 1940, Step: 42, State: 0, Action: IZQUIERDA, Reward: 0.0, Next State: 4
Q[0, 0]: 0.30593874064782084
Episode: 1940, Step: 43, State: 4, Action: IZQUIERDA, Reward: 0.0, Next State: 4
Q[4, 0]: 0.3212088145575592
Episode: 1940, Step: 44, State: 4, Action: IZQUIERDA, Reward: 0.0, Next State: 8
Q[4, 0]: 0.3255913768944546
Episode: 1940, Step: 45, State: 8, Action: ARRIBA, Reward: 0.0, Next State: 9
Q[8, 3]: 0.3748038992925613
Episode: 1940, Step: 46, State: 9, Action: ABAJO, Reward: 0.0, Next State: 10
Q[9, 1]: 0.4287911085285009
Episode: 1940, Step: 47, State: 10, Action: IZQUIERDA, Reward: 0.0, Next State: 14
Q[10, 0]: 0.4348774810137417
Episode: 1940, Step: 48, State: 14, Action: ABAJO, Reward: 0.0, Next State: 14
Q[14, 1]: 0.8751619039472501
Episode: 1940, Step: 49, State: 14, Action: ABAJO, Reward: 0.0, Next 

**Durante el Entrenamiento:**
   - El agente interactúa con el entorno (Frozen Lake) durante muchos episodios.
   - En cada paso, el agente toma una acción (izquierda, abajo, derecha, arriba).
   - La acción se decide usando la política epsilon-greedy:
      - Con probabilidad `epsilon`, el agente toma una acción aleatoria (exploración) para descubrir nuevas estrategias.
      - Con probabilidad `1-epsilon`, el agente toma la mejor acción según la tabla Q (explotación) basada en lo que ya ha aprendido.
   - Después de cada acción, el agente recibe una recompensa (1 si llega a la meta, -1 si cae en un agujero, 0 si no hace nada).
   - La tabla Q se actualiza usando la fórmula Q-learning: se mejora la estimación de la calidad de la acción tomada en ese estado.


**¿Qué nos dicen los resultados?**

- **Tabla Q:** Durante el entrenamiento, la tabla Q se llena con valores que representan la calidad de tomar una acción específica en un estado determinado.
- **Recompensas:** La lista `rewards` recopila la recompensa total obtenida en cada episodio.
- **Epsilon:** `epsilon` disminuye gradualmente a lo largo del entrenamiento, lo que hace que el agente explore menos y explote más.

**En esencia, se observa cómo el agente mejora su rendimiento a medida que aprende:**

- Al principio, el agente toma acciones aleatorias y la recompensa es baja.
- Con el tiempo, la tabla Q se llena con información útil y el agente toma decisiones más inteligentes.
- `epsilon` se reduce gradualmente, lo que permite que el agente siga explorando, pero al mismo tiempo aproveche su conocimiento de las mejores acciones.
- La recompensa aumenta porque el agente está tomando decisiones más correctas.

####**Paso #6: Mostrar los resultados**

Imprimimos la tabla Q y graficamos las recompensas acumuladas.

In [26]:
# Mostrar resultados
print("Tabla Q:")
print(Q)


Tabla Q:
[[ 0.39983396  0.18418704  0.24369244  0.20241949]
 [-0.05892554 -0.12712972 -0.10291618  0.2674823 ]
 [-0.14766354 -0.16486818 -0.1317861   0.16385905]
 [-0.20022227 -0.24865892 -0.51932485  0.18043102]
 [ 0.40994728  0.00736503 -0.01692293 -0.10149287]
 [ 0.          0.          0.          0.        ]
 [-0.63806261 -0.63601271  0.1046061  -0.60226109]
 [ 0.          0.          0.          0.        ]
 [-0.07655132 -0.12172589 -0.1821629   0.43462171]
 [-0.16009092  0.5194161   0.19940041 -0.21105773]
 [ 0.57604479 -0.1        -0.10449509 -0.14695546]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]
 [ 0.01347193  0.07051587  0.75691944 -0.15876483]
 [ 0.28732535  0.88547871  0.34709895  0.43432515]
 [ 0.          0.          0.          0.        ]]


In [ ]:
#Grafica de las recompensas acuumuladas

plt.figure(figsize=(10, 5))
plt.plot(rewards, label='Recompensa por episodio')
plt.xlabel('Episodio')
plt.ylabel('Recompensa')
plt.title('Evolución del Aprendizaje del Agente')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Crear una gráfica de dispersión para visualizar la relación entre el episodio y la recompensa
plt.figure(figsize=(10, 5))
plt.scatter(range(len(rewards)), rewards, s=5, label='Recompensa por episodio')
plt.xlabel('Episodio')
plt.ylabel('Recompensa')
plt.title('Evolución del Aprendizaje del Agente')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Crear una gráfica de barras para visualizar la distribución de las recompensas
plt.figure(figsize=(10, 5))
plt.bar(range(len(rewards)), rewards, label='Recompensa por episodio')
plt.xlabel('Episodio')
plt.ylabel('Recompensa')
plt.title('Evolución del Aprendizaje del Agente')
plt.grid(True)
plt.legend()
plt.show()

### **CONCLUSIONES:**

**En un contexto más general:**

El entrenamiento de Q-Learning busca que el agente encuentre la mejor estrategia para navegar en un entorno específico. Se busca que el agente maximice la recompensa total a largo plazo, aprendiendo de sus errores y las consecuencias de sus acciones.

**Tarea:**

Revisar un implementacion de Frozenlake en https://gymnasium.farama.org/tutorials/training_agents/FrozenLake_tuto/